# Лабораторная 6. Команда агентов проверяет договор
**AI Agents for Enterprise Processes Automation · Неделя 6**

Команда из трёх ИИ-агентов проверяет договор поставки, а вы сравниваете её работу со своей.

| Агент | Что делает | Кто пишет |
|---|---|---|
| Аналитик | читает договор и выписывает факты | готов |
| **Юрист** | ищет риски для Покупателя | **вы — Задание 1** |
| **Менеджер** | пишет заключение для директора | **вы — Задание 2** |

⚠️ **Лист группы (ручной анализ в аудитории) должен быть сдан ДО запуска этого ноутбука.**

Ячейки с пометкой «ГОТОВЫЙ КОД» просто запускайте (Shift+Enter) **по порядку, сверху вниз**. Ваша работа — в ячейках «ЗАДАНИЕ».

**Перед началом прочитайте:**
- Делайте ноутбук **за один заход (~1,5 ч)**. Через ~90 мин без активности Colab отключается и стирает всё, кроме текста ячеек: переменные, установленные программы, загруженные PDF. Тогда: выделите нужную ячейку → *Среда выполнения → Выполнить предшествующие*.
- Запуски команды пишутся с `await`: `e1 = await run(...)`. Без `await` ничего не произойдёт (предупреждение *coroutine was never awaited*).
- `NameError` — вы пропустили ячейку выше. Запустите её.
- **503 / UNAVAILABLE** — серверы Gemini перегружены, **429** — лимит бесплатного ключа. Ноутбук сам повторит 3 раза; если не помогло — смените модель в ячейке 0.3.
- Результаты LLM **меняются от запуска к запуску** — у соседа может получиться иначе, это нормально.
- В конце **скачайте `results.zip`** (ячейка Шага 5).

## Шаг 0. Подготовка

In [1]:
# Шаг 0.1. Установка (≈ 2–3 минуты). Запустите один раз.
# Предупреждения pip о конфликтах (gradio, google-adk и т. п.) — это нормально, их можно игнорировать.
!apt-get update -qq > /dev/null
!apt-get -qq install -y tesseract-ocr tesseract-ocr-rus poppler-utils > /dev/null
!tesseract --list-langs | grep -q rus && echo "Tesseract: русский язык есть" || echo "ОШИБКА: Tesseract без русского языка"
!pip -q install crewai pypdf pdf2image pytesseract google-genai rapidfuzz
print("Готово")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Tesseract: русский язык есть
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.5 MB/s e

In [2]:
# Шаг 0.2. Загрузите файлы договора: contract_digital.pdf и contract_scan.pdf
import os
from google.colab import files
os.makedirs("data", exist_ok=True)
os.makedirs("out", exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f"data/{name}", "wb") as f:
        f.write(content)
print("В папке data:", os.listdir("data"))

Saving contract_digital.pdf to contract_digital.pdf
В папке data: ['contract_digital.pdf']


In [3]:
# Шаг 0.2. Загрузите файлы договора: contract_digital.pdf и contract_scan.pdf
import os
from google.colab import files
os.makedirs("data", exist_ok=True)
os.makedirs("out", exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f"data/{name}", "wb") as f:
        f.write(content)
print("В папке data:", os.listdir("data"))

Saving contract_scan.pdf to contract_scan.pdf
В папке data: ['contract_digital.pdf', 'contract_scan.pdf']


In [4]:
# Шаг 0.3. Ключ Gemini. Добавьте его слева: значок ключа (Secrets) → имя GEMINI_API_KEY → включите доступ для ноутбука.
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
GEMINI_MODEL = "gemini-3.7-flash"   # ошибка 503 (модель перегружена) → "gemini-3.5-flash" или "gemini-flash-latest",
                                    # затем перезапустите эту ячейку и все ячейки ниже до запуска
print("Ключ найден")

Ключ найден


## Шаг 2. Как машина «читает» документ
Три способа из лекции: текстовый слой, OCR, Vision. Запустите и посмотрите, что получается со сканом.

In [5]:
# ГОТОВЫЙ КОД. Три способа прочитать документ (из лекции, раздел 5.2).
import base64, re
import pytesseract
from pypdf import PdfReader
from pdf2image import convert_from_path
from google import genai

def read_text_layer(pdf_path):
    """Текстовый слой PDF. У скана он пустой."""
    pages = [p.extract_text() or "" for p in PdfReader(pdf_path).pages]
    return "\n".join(f"--- стр. {i + 1} ---\n{t}" for i, t in enumerate(pages))

def ocr_pdf(pdf_path):
    """OCR: Tesseract распознаёт картинку страницы как текст (русский + английский)."""
    images = convert_from_path(pdf_path, dpi=200)
    return "\n".join(f"--- стр. {i + 1} ---\n{pytesseract.image_to_string(im, lang='rus+eng')}"
                     for i, im in enumerate(images))

VISION_PROMPT = """Перепиши весь текст этого документа в формате Markdown. Сохрани нумерацию пунктов и таблицы."""

def vision_read(pdf_path, prompt=None):
    """Vision: Gemini «смотрит» на страницы PDF и переписывает их в текст."""
    client = genai.Client()
    with open(pdf_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    interaction = client.interactions.create(
        model=GEMINI_MODEL,
        input=[{"type": "document", "data": data, "mime_type": "application/pdf"},
               {"type": "text", "text": prompt or VISION_PROMPT}],
    )
    return interaction.output_text

def has_text(text):
    return len(re.sub(r"--- стр\. \d+ ---|\s", "", text)) > 200

def read_document(pdf_path, mode):
    """mode = 'text' | 'ocr' | 'vision'.
    Возвращает (текст для агентов, эталонный текст для проверки цитат)."""
    layer = read_text_layer(pdf_path)
    if not has_text(layer):
        print("🔎 Текстового слоя нет — запускаем OCR для проверки цитат (≈30 секунд)…")
    deterministic = layer if has_text(layer) else ocr_pdf(pdf_path)
    if mode == "text":
        return layer, layer
    if mode == "ocr":
        return deterministic, deterministic
    print("👁  Gemini читает скан (Vision), это 30–90 секунд…")
    seen = vision_read(pdf_path)
    print(f"👁  Готово: {len(seen)} символов. Запускаем агентов…")
    return seen, deterministic + "\n" + seen

In [6]:
# Посмотрите, что видит каждый способ (без агентов). Сравните начало текстов.
digital = read_text_layer("data/contract_digital.pdf")
scan_layer = read_text_layer("data/contract_scan.pdf")
print("Цифровой PDF, текстовый слой:", len(digital), "символов")
print("Скан, текстовый слой:", len(scan_layer), "символов  ← почти пусто!")
scan_ocr = ocr_pdf("data/contract_scan.pdf")
print("Скан через OCR:", len(scan_ocr), "символов\n")
print(scan_ocr[:1500])

Цифровой PDF, текстовый слой: 6761 символов
Скан, текстовый слой: 63 символов  ← почти пусто!
Скан через OCR: 6770 символов

--- стр. 1 ---
ДОГОВОР № 47/2026

поставки оборудования и сервисного обслуживания
г. Бишкек «05» октября 2026 г.

ОсОО «ТехноСнаб Плюс», именуемое в дальнейшем «Поставщик», в лице директора
Асанова Т. K., действующего на основании Устава, с одной стороны, и ОсОО «Бирлик
Логистикс», именуемое в дальнейшем «Покупатель», в лице генерального директора
Омуралиевой А. Б., действующей на основании Устава, с другой стороны, совместно
именуемые «Стороны», заключили настоящий Договор о нижеследующем.

1. ПРЕДМЕТ ДОГОВОРА

1.1. Поставщик обязуется передать в собственность Покупателя складское
оборудование (далее — «Товар») согласно Спецификации (Приложение № 1), а также
оказывать услуги по сервисному обслуживанию Товара, а Покупатель обязуется
принять и оплатить Товар и услуги.

1.2. Товар поставляется новым, не бывшим в употреблении, в заводской упаковке.

2. ЦЕНА ТОВАРА



## Шаг 3. Команда агентов
Схемы данных, проверка цитат и Аналитик уже готовы.

In [7]:
# ГОТОВЫЙ КОД. Схемы данных: агенты передают друг другу не «текст», а проверяемые объекты.
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class ContractFacts(BaseModel):
    contract_no: Optional[str] = Field(None, description="Номер договора")
    date: Optional[str] = Field(None, description="Дата договора")
    supplier: Optional[str] = None
    buyer: Optional[str] = None
    total_amount_digits: Optional[str] = Field(None, description="Сумма цифрами, как в тексте")
    total_amount_words: Optional[str] = Field(None, description="Сумма прописью, как в тексте")
    prepayment_percent: Optional[str] = None
    payment_term: Optional[str] = Field(None, description="Срок оплаты")
    delivery_term: Optional[str] = Field(None, description="Срок поставки")
    handwritten_notes: List[str] = Field(default_factory=list, description="Рукописные правки, если видны")
    signatures_and_stamps: Optional[str] = Field(None, description="Чьи подписи и печати есть / отсутствуют")

Category = Literal["цена", "оплата", "поставка и приёмка", "ответственность", "форс-мажор",
                   "гарантия и сервис", "споры", "конфиденциальность", "оформление и подписи"]

class Risk(BaseModel):
    clause: str = Field(description="Номер пункта, например '6.2'")
    quote: str = Field(description="ДОСЛОВНАЯ цитата из договора, до 30 слов")
    category: Category
    severity: Literal["высокий", "средний", "низкий"]
    explanation: str = Field(description="Почему это риск для Покупателя")
    recommendation: str = Field(description="Какую правку предложить")

class RiskReport(BaseModel):
    risks: List[Risk]

In [8]:
# ГОТОВЫЙ КОД. Защита от выдуманных цитат (guardrail): проверяет КОД, а не ещё одна LLM.
from rapidfuzz import fuzz

REFERENCE_TEXT = ""   # заполняется автоматически при запуске

def normalize(s):
    s = re.sub(r"[*~`#|>]", " ", s.lower())   # убираем разметку Markdown из Vision
    s = re.sub(r"_+", "_", s)                  # «№ _____» и «№ _________» считаем одинаковыми
    return re.sub(r"\s+", " ", s).strip()

def quote_score(quote, source):
    """Цитату из нескольких кусков («... ») проверяем по кускам; берём худший."""
    parts = [p for p in re.split(r"\.\.\.|…", quote) if len(p.strip()) >= 15] or [quote]
    return min(fuzz.partial_ratio(normalize(p), source) for p in parts)

def quotes_exist(result):
    report = result.pydantic
    if report is None:
        return (False, "Ответ не соответствует схеме RiskReport. Верни список рисков по схеме.")
    source = normalize(REFERENCE_TEXT)
    missing = [r.clause for r in report.risks if quote_score(r.quote, source) < 80]
    if missing:
        return (False, f"Цитаты к пунктам {missing} не найдены в договоре. "
                       "Используй только дословные фрагменты текста договора.")
    return (True, result)

In [9]:
# ГОТОВЫЙ КОД. Модель и агент №1 — Аналитик.
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model=f"gemini/{GEMINI_MODEL}", temperature=0.1)

analyst = Agent(
    role="Аналитик входящих документов",
    goal="Точно извлечь ключевые факты договора",
    backstory=("Вы регистрируете входящие договоры в отделе закупок. Вы переносите данные как есть "
               "и никогда не додумываете: если значения нет в тексте — оставляете null. "
               "Если сумма цифрами и прописью различаются — записываете обе как в тексте."),
    llm=llm, allow_delegation=False, max_rpm=10,
)

extract_task = Task(
    description="Вот текст договора:\n\n{contract_text}\n\nИзвлеки ключевые факты договора.",
    expected_output="Объект ContractFacts; отсутствующие значения — null",
    agent=analyst,
    output_pydantic=ContractFacts,
)

### Задание 1. Юрист
Напишите `role`, `goal`, `backstory` и задачу. Подсказки:
- Юрист работает **на стороне Покупателя** — без этого модель пишет «нейтральный» анализ и пропускает односторонние условия.
- Перечислите в `backstory` **категории** рисков, которые нужно проверить (цена, оплата, приёмка, неустойки, форс-мажор, споры, подписи…), но **не номера пунктов** этого договора: команда должна работать и на другом договоре.
- Потребуйте **дословную цитату** для каждого риска — её проверит guardrail.
- Скажите, что выгодные Покупателю условия рисками не считаются.

In [10]:
lawyer = Agent(
    role="Старший юрисконсульт по защите интересов Покупателя",
    goal="Выявить все юридические и финансовые риски в договоре, защищая интересы Покупателя (ОсОО «Бирлик Логистикс»)",
    backstory=(
        "Вы опытный корпоративный юрист. Вы защищаете исключительно Покупателя. "
        "Ваша задача — тщательно проверить договор по всем категориям: цена, порядок расчётов, "
        "сроки поставки и приёмка, неустойки, ответственность, форс-мажор, гарантия, подсудность, подписи. "
        "Условия, выгодные Покупателю, рисками не считаются. "
        "Для каждого риска обязательна дословная цитата из текста договора (до 30 слов)."
    ),
    llm=llm,
    allow_delegation=False,
    max_rpm=10,
)

risk_task = Task(
    description=(
        "Вот текст договора:\n\n{contract_text}\n\n"
        "Проанализируй договор и найди все невыгодные и рискованные условия для Покупателя. "
        "Для каждого риска укажи номер пункта, ДОСЛОВНУЮ цитату, категорию, степень суровости (высокий/средний/низкий), "
        "объяснение и конкретную рекомендацию по правке."
    ),
    expected_output="Объект RiskReport, содержащий список всех найденных рисков по схеме Risk",
    agent=lawyer,
    context=[extract_task],
    output_pydantic=RiskReport,
    guardrail=quotes_exist,
    guardrail_max_retries=2,
)

### Задание 2. Менеджер
Менеджер пишет для директора, который **не юрист** и читает не больше одной страницы.
- Запретите ему добавлять новые факты и риски — только пересказывать выводы Аналитика и Юриста.
- В `expected_output` опишите формат по пунктам: вывод (ПОДПИСАТЬ / С ПРАВКАМИ / НЕ ПОДПИСЫВАТЬ), главные риски, правки, что проверить человеку.

In [11]:
manager = Agent(
    role="Операционный менеджер по согласованию договоров",
    goal="Подготовить лаконичное финальное заключение по договору для генерального директора",
    backstory=(
        "Вы готовите сводку для директора, у которого нет времени читать весь договор. "
        "Вы опираетесь СТРОГО на факты Аналитика и риски Юриста. Запрещено придумывать новые факты или риски. "
        "Пишите кратко, понятно, простым языком и по делу."
    ),
    llm=llm,
    allow_delegation=False,
    max_rpm=10,
)

report_task = Task(
    description=(
        "На основе найденных фактов и рисков подготовь итоговое заключение для директора. "
        "Сделай чёткий вывод о том, можно ли подписывать договор, выдели 3 главных риска "
        "и сформируй список критических правок."
    ),
    expected_output=(
        "Отчёт в формате Markdown:\n"
        "1. Решение (ПОДПИСАТЬ / ПОДПИСАТЬ С ПРАВКАМИ / НЕ ПОДПИСЫВАТЬ)\n"
        "2. Три главных риска простыми словами (с номерами пунктов)\n"
        "3. Что обязательно исправить перед подписанием\n"
        "4. Что проверить человеку (рукописные пометки, подписи и печати)"
    ),
    agent=manager,
    context=[extract_task, risk_task],
    markdown=True,
)

In [12]:
# ГОТОВЫЙ КОД. Запуск команды и сохранение результата.
import asyncio, json, time
import pandas as pd
from IPython.display import Markdown, display

async def run(pdf_name, mode, label):
    global REFERENCE_TEXT
    t0 = time.perf_counter()
    doc_text, REFERENCE_TEXT = read_document(f"data/{pdf_name}", mode)
    crew = Crew(agents=[analyst, lawyer, manager],
                tasks=[extract_task, risk_task, report_task],
                process=Process.sequential, verbose=True)
    for attempt in range(1, 4):   # до 3 попыток: облачные модели бывают перегружены (503) или упираются в лимит (429)
        try:
            result = await crew.kickoff_async(inputs={"contract_text": doc_text})  # в Colab нужен асинхронный запуск
            break
        except Exception as e:
            temporary = any(code in str(e) for code in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"))
            if not temporary or attempt == 3:
                raise
            print(f"\n⏳ Сервер Gemini занят или лимит исчерпан (попытка {attempt}/3). Ждём 40 секунд…")
            await asyncio.sleep(40)
    seconds = time.perf_counter() - t0
    facts = extract_task.output.pydantic if extract_task.output else None
    risks = risk_task.output.pydantic if risk_task.output else None
    log = {"label": label, "pdf": pdf_name, "mode": mode,
           "seconds": round(seconds, 1),
           "tokens": result.token_usage.total_tokens,
           "facts": facts.model_dump() if facts else None,
           "risks": [r.model_dump() for r in risks.risks] if risks else [],
           "report": result.raw}
    with open(f"out/{label}.json", "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)
    print(f"\n{label}: {seconds:.0f} с · токенов {log['tokens']} · найдено рисков {len(log['risks'])}")
    display(Markdown(result.raw))
    return log

def risks_table(log):
    return pd.DataFrame(log["risks"])[["clause", "severity", "category", "explanation"]]

## Шаг 4. Два запуска
**E1** — цифровой PDF (текстовый слой). **E2** — скан через Vision. Каждый запуск занимает 1–5 минут.

In [13]:
e1 = await run("contract_digital.pdf", "text", "E1_digital")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 16198abb-6488-4006-8ce5-9ee1dcf84bc5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Вот текст договора:                                                                                      │
│                                                                                                                 │
│  --- стр. 1 ---                                                                                                 │
│  Договор № 47/2026 · стр. 1                                                                                     │
│   ДОГОВОР № 47/2026                                                                                             │
│   поставки оборудования и сервисного обслуживания                                                               │
│  г. Бишкек                                                                                                      │
│  «05» октября 2026 г.                                                                                           │
│  ОсОО «ТехноСнаб Плюс», именуемое в дальнейшем «Поставщик», в лице директора                                    │
│  Асанова Т. К., действующего на основании Устава, с одной стороны, и ОсОО «Бирлик                               │
│  Логистикс», именуемое в дальнейшем «Покупатель», в лице генерального директора                                 │
│  Омуралиевой А. Б., действующей на основании Устава, с другой стороны, совместно                                │
│  именуемые «Стороны», заключили настоящий Договор о нижеследующем.                                              │
│  1. ПРЕДМЕТ ДОГОВОРА                                                                                            │
│  1.1. Поставщик обязуется передать в собственность Покупателя складское                                         │
│  оборудование (далее — «Товар») согласно Спецификации (Приложение № 1), а также                                 │
│  оказывать услуги по сервисному обслуживанию Товара, а Покупатель обязуется                                     │
│  принять и оплатить Товар и услуги.                                                                             │
│  1.2. Товар поставляется новым, не бывшим в употреблении, в заводской упаковке.                                 │
│  2. ЦЕНА ТОВАРА                                                                                                 │
│  2.1. Цена Товара указана в Спецификации и включает стоимость упаковки и                                        │
│  маркировки.                                                                                                    │
│  2.2. Расходы по доставке Товара до склада Покупателя (г. Бишкек, ул. Ахунбаева, 150)                           │
│  несёт Поставщик.                                                                                               │
│  2.3. Поставщик вправе в одностороннем порядке изменить цену Товара, в том числе                                │
│  уже оплаченного, в случае изменения официального курса доллара США более чем на                                │
│  3 (три) процента, уведомив Покупателя за 5 (пять) календарных дней. Согласие                                   │
│  Покупателя на такое изменение не требуется.                                                                    │
│  3. ПОРЯДОК РАСЧЁТОВ                                                                                            │
│  3.1. Покупатель производит предварительную оплату в размере 100 % (сто процентов)                              │
│  стоимости Товара в течение 3 (трёх) банковских дней с даты подписания настоящего                               │
│  Договора.                                             

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Аналитик входящих документов                                                                            │
│                                                                                                                 │
│  Task: Вот текст договора:                                                                                      │
│                                                                                                                 │
│  --- стр. 1 ---                                                                                                 │
│  Договор № 47/2026 · стр. 1                                                                                     │
│   ДОГОВОР № 47/2026                                                                                             │
│   поставки оборудования и сервисного обслуживания                                                               │
│  г. Бишкек                                                                                                      │
│  «05» октября 2026 г.                                                                                           │
│  ОсОО «ТехноСнаб Плюс», именуемое в дальнейшем «Поставщик», в лице директора                                    │
│  Асанова Т. К., действующего на основании Устава, с одной стороны, и ОсОО «Бирлик                               │
│  Логистикс», именуемое в дальнейшем «Покупатель», в лице генерального директора                                 │
│  Омуралиевой А. Б., действующей на основании Устава, с другой стороны, совместно                                │
│  именуемые «Стороны», заключили настоящий Договор о нижеследующем.                                              │
│  1. ПРЕДМЕТ ДОГОВОРА                                                                                            │
│  1.1. Поставщик обязуется передать в собственность Покупателя складское                                         │
│  оборудование (далее — «Товар») согласно Спецификации (Приложение № 1), а также                                 │
│  оказывать услуги по сервисному обслуживанию Товара, а Покупатель обязуется                                     │
│  принять и оплатить Товар и услуги.                                                                             │
│  1.2. Товар поставляется новым, не бывшим в употреблении, в заводской упаковке.                                 │
│  2. ЦЕНА ТОВАРА                                                                                                 │
│  2.1. Цена Товара указана в Спецификации и включает стоимость упаковки и                                        │
│  маркировки.                                                                                                    │
│  2.2. Расходы по доставке Товара до склада Покупателя (г. Бишкек, ул. Ахунбаева, 150)                           │
│  несёт Поставщик.                                                                                               │
│  2.3. Поставщик вправе в одностороннем порядке изменить цену Товара, в том числе                                │
│  уже оплаченного, в случае изменения официального курса доллара США более чем на                                │
│  3 (три) процента, уведомив Покупателя за 5 (пять) календарных дней. Согласие                                   │
│  Покупателя на такое изменение не требуется.                                                                    │
│  3. ПОРЯДОК РАСЧЁТОВ                                                                                            │
│  3.1. Покупатель производит предварительную оплату в размере 100 % (сто процентов)                              │
│  стоимости Товара в течение 3 (трёх) банковских дней с 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Аналитик входящих документов                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  contract_no='47/2026' date='«05» октября 2026 г.' supplier='ОсОО «ТехноСнаб Плюс»' buyer='ОсОО «Бирлик         │
│  Логистикс»' total_amount_digits='2 450 000' total_amount_words='два миллиона сорок пять тысяч'                 │
│  prepayment_percent='100 %' payment_term='в течение 3 (трёх) банковских дней с даты подписания настоящего       │
│  Договора' delivery_term='в течение 30 (тридцати) календарных дней с момента поступления предоплаты (с          │
│  возможностью переноса срока Поставщиком на период до 60 (шестидесяти) календарных дней)' handwritten_notes=[]  │
│  signatures_and_stamps='Подписи и печати отсутствуют (присутствуют только места для подписей Асанова Т. К. и    │
│  Омуралиевой А. Б., реквизиты доверенности представителя Поставщика не заполнены)'                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Вот текст договора:                                                                                      │
│                                                                                                                 │
│  --- стр. 1 ---                                                                                                 │
│  Договор № 47/2026 · стр. 1                                                                                     │
│   ДОГОВОР № 47/2026                                                                                             │
│   поставки оборудования и сервисного обслуживания                                                               │
│  г. Бишкек                                                                                                      │
│  «05» октября 2026 г.                                                                                           │
│  ОсОО «ТехноСнаб Плюс», именуемое в дальнейшем «Поставщик», в лице директора                                    │
│  Асанова Т. К., действующего на основании Устава, с одной стороны, и ОсОО «Бирлик                               │
│  Логистикс», именуемое в дальнейшем «Покупатель», в лице генерального директора                                 │
│  Омуралиевой А. Б., действующей на основании Устава, с другой стороны, совместно                                │
│  именуемые «Стороны», заключили настоящий Договор о нижеследующем.                                              │
│  1. ПРЕДМЕТ ДОГОВОРА                                                                                            │
│  1.1. Поставщик обязуется передать в собственность Покупателя складское                                         │
│  оборудование (далее — «Товар») согласно Спецификации (Приложение № 1), а также                                 │
│  оказывать услуги по сервисному обслуживанию Товара, а Покупатель обязуется                                     │
│  принять и оплатить Товар и услуги.                                                                             │
│  1.2. Товар поставляется новым, не бывшим в употреблении, в заводской упаковке.                                 │
│  2. ЦЕНА ТОВАРА                                                                                                 │
│  2.1. Цена Товара указана в Спецификации и включает стоимость упаковки и                                        │
│  маркировки.                                                                                                    │
│  2.2. Расходы по доставке Товара до склада Покупателя (г. Бишкек, ул. Ахунбаева, 150)                           │
│  несёт Поставщик.                                                                                               │
│  2.3. Поставщик вправе в одностороннем порядке изменить цену Товара, в том числе                                │
│  уже оплаченного, в случае изменения официального курса доллара США более чем на                                │
│  3 (три) процента, уведомив Покупателя за 5 (пять) календарных дней. Согласие                                   │
│  Покупателя на такое изменение не требуется.                                                                    │
│  3. ПОРЯДОК РАСЧЁТОВ                                                                                            │
│  3.1. Покупатель производит предварительную оплату в размере 100 % (сто процентов)                              │
│  стоимости Товара в течение 3 (трёх) банковских дней с даты подписания настоящего                               │
│  Договора.                                             

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Операционный менеджер по согласованию договоров                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Заключение по договору поставки № 47/2026 от 05.10.2026                                                      │
│  **Контрагент:** ОсОО «ТехноСнаб Плюс»                                                                          │
│  **Покупатель:** ОсОО «Бирлик Логистикс»                                                                        │
│  **Сумма договора:** 2 450 000 сом                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Решение                                                                                                 │
│  **ПОДПИСАТЬ С ПРАВКАМИ** *(в текущей редакции подписывать категорически нельзя — условия кабальные)*.          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Три главных риска простыми словами                                                                      │
│                                                                                                                 │
│  1. **Потеря денег и неконтролируемый рост цены (п. 2.3, п. 3.1)**                                              │
│     С нас требуют 100% предоплату в течение 3 дней, но при росте курса доллара более чем на 3% поставщик имеет  │
│  право в одностороннем порядке поднять цену даже на уже полностью оплаченный товар. Права отказаться от         │
│  договора без потерь у нас нет.                                                                                 │
│  2. **Срыв сроков без ответственности и несимметричные штрафы (п. 4.3, п. 6.1, п. 6.2, п. 6.4)**                │
│     Поставщик вправе задержать поставку на 60 дней сверх 30-дневного срока (суммарно до 90 дней) вообще без     │
│  штрафов, а его общая ответственность ограничена 5% от суммы договора. При этом с нас за каждый день просрочки  │
│  оплаты требуют грабительскую неустойку 0,5% в день (182,5% годовых) без какого-либо лимита.                    │
│  3. **Блокировка судебной защиты (п. 10.2)**                                                                    │
│     Все споры перенесены в Международный арбитражный центр в Дубае (ОАЭ) на английском языке с оплатой всех     │
│  расходов истцом. Защитить свои права при цене договора в 2,45 млн сом будет невозможно — расходы на юристов и  │
│  поездки в Дубай превысят сумму контракта.                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Что обязательно исправить перед подписанием    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: На основе найденных фактов и рисков подготовь итоговое заключение для директора. Сделай чёткий вывод о   │
│  том, можно ли подписывать договор, выдели 3 главных риска и сформируй список критических правок.               │
│  Agent: Операционный менеджер по согласованию договоров                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 16198abb-6488-4006-8ce5-9ee1dcf84bc5                                                                       │
│  Final Output: # Заключение по договору поставки № 47/2026 от 05.10.2026                                        │
│  **Контрагент:** ОсОО «ТехноСнаб Плюс»                                                                          │
│  **Покупатель:** ОсОО «Бирлик Логистикс»                                                                        │
│  **Сумма договора:** 2 450 000 сом                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Решение                                                                                                 │
│  **ПОДПИСАТЬ С ПРАВКАМИ** *(в текущей редакции подписывать категорически нельзя — условия кабальные)*.          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Три главных риска простыми словами                                                                      │
│                                                                                                                 │
│  1. **Потеря денег и неконтролируемый рост цены (п. 2.3, п. 3.1)**                                              │
│     С нас требуют 100% предоплату в течение 3 дней, но при росте курса доллара более чем на 3% поставщик имеет  │
│  право в одностороннем порядке поднять цену даже на уже полностью оплаченный товар. Права отказаться от         │
│  договора без потерь у нас нет.                                                                                 │
│  2. **Срыв сроков без ответственности и несимметричные штрафы (п. 4.3, п. 6.1, п. 6.2, п. 6.4)**                │
│     Поставщик вправе задержать поставку на 60 дней сверх 30-дневного срока (суммарно до 90 дней) вообще без     │
│  штрафов, а его общая ответственность ограничена 5% от суммы договора. При этом с нас за каждый день просрочки  │
│  оплаты требуют грабительскую неустойку 0,5% в день (182,5% годовых) без какого-либо лимита.                    │
│  3. **Блокировка судебной защиты (п. 10.2)**                                                                    │
│     Все споры перенесены в Международный арбитражный центр в Дубае (ОАЭ) на английском языке с оплатой всех     │
│  расходов истцом. Защитить свои права при цене договора в 2,45 млн сом будет невозможно — расходы на юристов и  │
│  поездки в Дубай превысят сумму контракта.                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Что обязательно исправить перед подписанием   


E1_digital: 302 с · токенов 69963 · найдено рисков 16


# Заключение по договору поставки № 47/2026 от 05.10.2026
**Контрагент:** ОсОО «ТехноСнаб Плюс»  
**Покупатель:** ОсОО «Бирлик Логистикс»  
**Сумма договора:** 2 450 000 сом  

---

### 1. Решение
**ПОДПИСАТЬ С ПРАВКАМИ** *(в текущей редакции подписывать категорически нельзя — условия кабальные)*.

---

### 2. Три главных риска простыми словами

1. **Потеря денег и неконтролируемый рост цены (п. 2.3, п. 3.1)**  
   С нас требуют 100% предоплату в течение 3 дней, но при росте курса доллара более чем на 3% поставщик имеет право в одностороннем порядке поднять цену даже на уже полностью оплаченный товар. Права отказаться от договора без потерь у нас нет.
2. **Срыв сроков без ответственности и несимметричные штрафы (п. 4.3, п. 6.1, п. 6.2, п. 6.4)**  
   Поставщик вправе задержать поставку на 60 дней сверх 30-дневного срока (суммарно до 90 дней) вообще без штрафов, а его общая ответственность ограничена 5% от суммы договора. При этом с нас за каждый день просрочки оплаты требуют грабительскую неустойку 0,5% в день (182,5% годовых) без какого-либо лимита.
3. **Блокировка судебной защиты (п. 10.2)**  
   Все споры перенесены в Международный арбитражный центр в Дубае (ОАЭ) на английском языке с оплатой всех расходов истцом. Защитить свои права при цене договора в 2,45 млн сом будет невозможно — расходы на юристов и поездки в Дубай превысят сумму контракта.

---

### 3. Что обязательно исправить перед подписанием

* **Пункт 3.2 (Ошибка в сумме):** Устранить техническую ошибку — прописью указано *«два миллиона сорок пять тысяч»* (2 045 000), а цифрами — `2 450 000`. Изложить прописью: *«два миллиона четыреста пятьдесят тысяч сом»*.
* **Пункт 2.3 (Цена):** Исключить право поставщика менять цену в одностороннем порядке. Зафиксировать твердую цену в национальной валюте (сомах).
* **Пункт 3.1 (Оплата):** Заменить 100% предоплату на частичную (например, 20–30%), а остаток выплачивать после фактической поставки и подписания акта приемки.
* **Пункт 4.3 (Сроки поставки):** Полностью исключить пункт о праве поставщика переносить срок поставки на 60 дней без штрафов.
* **Пункт 5.2 (Приёмка):** Увеличить срок приемки и направления мотивированного отказа с 1 рабочего дня до 5–10 рабочих дней. Исключить условие о том, что скрытые дефекты признаются принятыми Покупателем.
* **Пункты 6.1, 6.2 и 6.4 (Ответственность):** 
  - Снизить неустойку Покупателя за просрочку оплаты с 0,5% до 0,05–0,1% в день и ограничить её лимитом (не более 5–10% от долга).
  - Повысить ответственность Поставщика за срыв поставки до паритетных 0,1% в день.
  - Исключить лимит совокупной ответственности Поставщика в 5%.
* **Пункт 7.1 (Форс-мажор):** Убрать из форс-мажора валютные колебания, санкции и формулировку *«по усмотрению Поставщика»*. Ввести обязательное подтверждение свидетельством ТПП.
* **Пункт 10.2 (Подсудность):** Заменить арбитраж Дубая на суды Кыргызской Республики по месту нахождения Покупателя либо Международный третейский суд при ТПП КР (г. Бишкек).
* **Пункт 13 и преамбула:** Привести к единообразию данные подписанта со стороны Поставщика (в шапке указан директор на основании Устава, в конце — представитель по доверенности).

---

### 4. Что проверить человеку перед визированием

* **Рукописные пометки:** В переданном проекте рукописных записей нет. При получении бумажного экземпляра убедиться в отсутствии пометок, исправлений и дописок на полях от руки.
* **Подписи и печати:** В предоставленном проекте подписи и печати сторон **отсутствуют**. Требуется проследить за их простановкой на всех страницах/сшивке.
* **Полномочия подписанта Поставщика:** В разделе 13 указан представитель по доверенности Асанов Т. К., но **номер и дата доверенности не заполнены**. До подписания необходимо:
  1. Запросить и проверить оригинал доверенности Асанова Т. К. (срок действия, право заключения договоров на сумму свыше 2,45 млн сом).
  2. Вписать фактические реквизиты доверенности (номер и дату) в текст договора.

In [14]:
risks_table(e1)

,clause,severity,category,explanation
0,2.3,высокий,цена,Поставщик может в одностороннем порядке увелич...
1,3.1,высокий,оплата,Условие о 100% предоплате замораживает оборотн...
2,3.2,средний,цена,Прямое несоответствие суммы цифрами и прописью...
3,3.4,низкий,оплата,Покупатель несёт риски банковских задержек и о...
4,4.3,высокий,поставка и приёмка,Поставщик может в одностороннем порядке задерж...
5,5.2,высокий,поставка и приёмка,"Одного рабочего дня недостаточно для монтажа, ..."
6,6.1,высокий,ответственность,"Кабальная неустойка (0,5% в день = 182,5% годо..."
7,6.2,высокий,ответственность,"Ответственность Поставщика в 50 раз меньше, че..."
8,6.4,высокий,ответственность,Лимит совокупной ответственности в 5% лишает П...
9,7.1,высокий,форс-мажор,Колебания валюты относятся к обычным предприни...


In [15]:
e2 = await run("contract_scan.pdf", "vision", "E2_scan_vision")

🔎 Текстового слоя нет — запускаем OCR для проверки цитат (≈30 секунд)…
👁  Gemini читает скан (Vision), это 30–90 секунд…
👁  Готово: 7044 символов. Запускаем агентов…


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 59577f16-5ff4-4f19-97d6-380466aa314d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Вот текст договора:                                                                                      │
│                                                                                                                 │
│  ### ДОГОВОР № 47/2026                                                                                          │
│  **поставки оборудования и сервисного обслуживания**                                                            │
│                                                                                                                 │
│  **г. Бишкек**                                                                                                  │
│  **«05» октября 2026 г.**                                                                                       │
│                                                                                                                 │
│  ОсОО «ТехноСнаб Плюс», именуемое в дальнейшем «Поставщик», в лице директора Асанова Т. К., действующего на     │
│  основании Устава, с одной стороны, и ОсОО «Бирлик Логистикс», именуемое в дальнейшем «Покупатель», в лице      │
│  генерального директора Омуралиевой А. Б., действующей на основании Устава, с другой стороны, совместно         │
│  именуемые «Стороны», заключили настоящий Договор о нижеследующем.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### 1. ПРЕДМЕТ ДОГОВОРА                                                                                       │
│                                                                                                                 │
│  1.1. Поставщик обязуется передать в собственность Покупателя складское оборудование (далее — «Товар»)          │
│  согласно Спецификации (Приложение № 1), а также оказывать услуги по сервисному обслуживанию Товара, а          │
│  Покупатель обязуется принять и оплатить Товар и услуги.                                                        │
│  1.2. Товар поставляется новым, не бывшим в употреблении, в заводской упаковке.                                 │
│                                                                                                                 │
│  #### 2. ЦЕНА ТОВАРА                                                                                            │
│                                                                                                                 │
│  2.1. Цена Товара указана в Спецификации и включает стоимость упаковки и маркировки.                            │
│  2.2. Расходы по доставке Товара до склада Покупателя (г. Бишкек, ул. Ахунбаева, 150) несёт Поставщик.          │
│  2.3. Поставщик вправе в одностороннем порядке изменить цену Товара, в том числе уже оплаченного, в случае      │
│  изменения официального курса доллара США более чем на 3 (три) процента, уведомив Покупателя за 5 (пять)        │
│  календарных дней. Согласие Покупателя на такое изменение не требуется.                                         │
│                                                                                                                 │
│  #### 3. ПОРЯДОК РАСЧЁТОВ                                                                                       │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Аналитик входящих документов                                                                            │
│                                                                                                                 │
│  Task: Вот текст договора:                                                                                      │
│                                                                                                                 │
│  ### ДОГОВОР № 47/2026                                                                                          │
│  **поставки оборудования и сервисного обслуживания**                                                            │
│                                                                                                                 │
│  **г. Бишкек**                                                                                                  │
│  **«05» октября 2026 г.**                                                                                       │
│                                                                                                                 │
│  ОсОО «ТехноСнаб Плюс», именуемое в дальнейшем «Поставщик», в лице директора Асанова Т. К., действующего на     │
│  основании Устава, с одной стороны, и ОсОО «Бирлик Логистикс», именуемое в дальнейшем «Покупатель», в лице      │
│  генерального директора Омуралиевой А. Б., действующей на основании Устава, с другой стороны, совместно         │
│  именуемые «Стороны», заключили настоящий Договор о нижеследующем.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### 1. ПРЕДМЕТ ДОГОВОРА                                                                                       │
│                                                                                                                 │
│  1.1. Поставщик обязуется передать в собственность Покупателя складское оборудование (далее — «Товар»)          │
│  согласно Спецификации (Приложение № 1), а также оказывать услуги по сервисному обслуживанию Товара, а          │
│  Покупатель обязуется принять и оплатить Товар и услуги.                                                        │
│  1.2. Товар поставляется новым, не бывшим в употреблении, в заводской упаковке.                                 │
│                                                                                                                 │
│  #### 2. ЦЕНА ТОВАРА                                                                                            │
│                                                                                                                 │
│  2.1. Цена Товара указана в Спецификации и включает стоимость упаковки и маркировки.                            │
│  2.2. Расходы по доставке Товара до склада Покупателя (г. Бишкек, ул. Ахунбаева, 150) несёт Поставщик.          │
│  2.3. Поставщик вправе в одностороннем порядке изменить цену Товара, в том числе уже оплаченного, в случае      │
│  изменения официального курса доллара США более чем на 3 (три) процента, уведомив Покупателя за 5 (пять)        │
│  календарных дней. Согласие Покупателя на такое изменение не требуется.                                         │
│                                                                                                                 │
│  #### 3. ПОРЯДОК РАСЧЁТОВ                              

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Аналитик входящих документов                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  contract_no='47/2026' date='«05» октября 2026 г.' supplier='ОсОО «ТехноСнаб Плюс»' buyer='ОсОО «Бирлик         │
│  Логистикс»' total_amount_digits='2 450 000' total_amount_words='два миллиона сорок пять тысяч'                 │
│  prepayment_percent='100 %' payment_term='в течение 3 (трёх) банковских дней с даты подписания настоящего       │
│  Договора (исправлено на: (один) банк. день!)' delivery_term='в течение 30 (тридцати) календарных дней с        │
│  момента поступления предоплаты' handwritten_notes=['(один) банк. день! / согласовано /А.О./ 06.10.26']         │
│  signatures_and_stamps='Поставщик: подпись и печать отсутствуют. Покупатель: есть подпись Омуралиевой А. Б. и   │
│  печать ОсОО «БИРЛИК ЛОГИСТИКС» г. БИШКЕК'                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Операционный менеджер по согласованию договоров                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Заключение по договору поставки № 47/2026 от 05.10.2026                                                      │
│  **Контрагент:** ОсОО «ТехноСнаб Плюс» (Поставщик)                                                              │
│  **Покупатель:** ОсОО «Бирлик Логистикс»                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Решение                                                                                                 │
│  **ПОДПИСАТЬ С ПРАВКАМИ** *(в текущей редакции подписывать категорически нельзя).*                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Три главных риска простыми словами                                                                      │
│  1. **Финансовая ловушка и потеря денег (п. 2.3, п. 3.1, п. 3.2):**                                             │
│     В договоре критическая ошибка в сумме: цифрами указано `2 450 000` сом, а прописью — `два миллиона сорок    │
│  пять тысяч` (разница 405 000 сом). При этом с нас требуют 100% предоплату, а Поставщик оставляет за собой      │
│  право поднять цену уже после нашей оплаты, если курс доллара вырастет всего на 3%.                             │
│  2. **Безнаказанные задержки и отсутствие гарантий (п. 4.3, п. 5.2, п. 6.2, п. 6.4):**                          │
│     Поставщик вправе задержать поставку на 60 дней без штрафов и объяснений. На проверку товара нам дают всего  │
│  1 рабочий день, после чего претензии по скрытому браку не принимаются. Ответственность Поставщика за срыв      │
│  сроков мизерная (0,01%, максимум 2%) и в целом ограничена 5% от суммы, тогда как с нас за день просрочки       │
│  требуют грабительские 0,5% (более 180% годовых).                                                               │
│  3. **Судебные споры в Дубае (п. 10.2):**                                                                       │
│     Все разногласия двух бишкекских компаний переданы в Международный арбитраж Дубая (ОАЭ) на английском        │
│  языке. Расходы на перелеты, пошлины и иностранных юристов в случае спора многократно превысят стоимость        │
│  самого договора.                                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Что обязательно исправить перед подписанием                                                             │
│  - **Сумма договора (п. 3.2):** Устранить расхождение, 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: На основе найденных фактов и рисков подготовь итоговое заключение для директора. Сделай чёткий вывод о   │
│  том, можно ли подписывать договор, выдели 3 главных риска и сформируй список критических правок.               │
│  Agent: Операционный менеджер по согласованию договоров                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


E2_scan_vision: 711 с · токенов 137361 · найдено рисков 16


# Заключение по договору поставки № 47/2026 от 05.10.2026
**Контрагент:** ОсОО «ТехноСнаб Плюс» (Поставщик)  
**Покупатель:** ОсОО «Бирлик Логистикс»  

---

### 1. Решение
**ПОДПИСАТЬ С ПРАВКАМИ** *(в текущей редакции подписывать категорически нельзя).*

---

### 2. Три главных риска простыми словами
1. **Финансовая ловушка и потеря денег (п. 2.3, п. 3.1, п. 3.2):**  
   В договоре критическая ошибка в сумме: цифрами указано `2 450 000` сом, а прописью — `два миллиона сорок пять тысяч` (разница 405 000 сом). При этом с нас требуют 100% предоплату, а Поставщик оставляет за собой право поднять цену уже после нашей оплаты, если курс доллара вырастет всего на 3%.
2. **Безнаказанные задержки и отсутствие гарантий (п. 4.3, п. 5.2, п. 6.2, п. 6.4):**  
   Поставщик вправе задержать поставку на 60 дней без штрафов и объяснений. На проверку товара нам дают всего 1 рабочий день, после чего претензии по скрытому браку не принимаются. Ответственность Поставщика за срыв сроков мизерная (0,01%, максимум 2%) и в целом ограничена 5% от суммы, тогда как с нас за день просрочки требуют грабительские 0,5% (более 180% годовых).
3. **Судебные споры в Дубае (п. 10.2):**  
   Все разногласия двух бишкекских компаний переданы в Международный арбитраж Дубая (ОАЭ) на английском языке. Расходы на перелеты, пошлины и иностранных юристов в случае спора многократно превысят стоимость самого договора.

---

### 3. Что обязательно исправить перед подписанием
- **Сумма договора (п. 3.2):** Устранить расхождение, прямо указать: *«2 450 000 (два миллиона четыреста пятьдесят тысяч) сом»*.
- **Условия оплаты и цены (п. 2.3, п. 3.1, п. 3.4):** 
  - Исключить право Поставщика на одностороннее повышение цены (цена должна быть твердой).
  - Разбить оплату: аванс не более 30–50%, остаток — строго после поставки и приемки.
  - Моментом оплаты считать дату списания средств с нашего расчетного счета, а не зачисления Поставщику.
- **Сроки и приемка (п. 4.3, п. 5.2):** 
  - Исключить пункт 4.3 (срок поставки должен быть жестким, перенос — только по соглашению сторон).
  - Увеличить срок приемки товара до 5–10 рабочих дней; полностью убрать отказ от претензий по скрытым недостаткам.
- **Баланс ответственности (п. 6.1, п. 6.2, п. 6.4):** 
  - Уравнять неустойки: снизить штраф Покупателя до 0,05–0,1% в день (с лимитом не более 5–10%) и установить зеркальный штраф для Поставщика.
  - Исключить ограничение ответственности Поставщика планкой в 5%.
- **Форс-мажор и сервис (п. 7.1, п. 8.2, п. 9.1):** 
  - Убрать из форс-мажора колебания курса, санкции третьих лиц и право определять форс-мажор «по усмотрению Поставщика».
  - Зафиксировать тарифы на гарантийный сервис в договоре и сократить срок отказа от пролонгации сервиса с 90 до 30 дней.
- **Подсудность (п. 10.2):** Изменить место рассмотрения споров на компетентный суд Кыргызской Республики (г. Бишкек) на русском языке.
- **Маркетинг (п. 11.2):** Исключить право Поставщика свободно использовать наш логотип и бренд без письменного согласия.

---

### 4. Что проверить человеку (рукописные пометки, подписи и печати)
- **Подписи и печати Поставщика (раздел 13):** Со стороны ОсОО «ТехноСнаб Плюс» договор **не подписан** — подпись и печать отсутствуют. 
- **Полномочия подписанта (преамбула и раздел 13):** В начале договора указан директор Асанов Т.К. на основании Устава, а в разделе 13 стоит подпись по доверенности, реквизиты которой не заполнены (`№ _____ от _________`). Необходимо привести преамбулу и раздел 13 к единому основанию и проверить документы на подписанта.
- **Рукописная пометка (п. 3.1):** В пункте о сроке оплаты от руки вписано со стрелкой: `(один) банк. день! / согласовано /А.О./ 06.10.26`. Отметка согласована только подписью Покупателя (А.О.), со стороны Поставщика подтверждения нет. Договор необходимо перепечатать начисто с безопасным сроком оплаты (не менее 3–5 банковских дней), убрав любые рукописные правки.

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 59577f16-5ff4-4f19-97d6-380466aa314d                                                                       │
│  Final Output: # Заключение по договору поставки № 47/2026 от 05.10.2026                                        │
│  **Контрагент:** ОсОО «ТехноСнаб Плюс» (Поставщик)                                                              │
│  **Покупатель:** ОсОО «Бирлик Логистикс»                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Решение                                                                                                 │
│  **ПОДПИСАТЬ С ПРАВКАМИ** *(в текущей редакции подписывать категорически нельзя).*                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Три главных риска простыми словами                                                                      │
│  1. **Финансовая ловушка и потеря денег (п. 2.3, п. 3.1, п. 3.2):**                                             │
│     В договоре критическая ошибка в сумме: цифрами указано `2 450 000` сом, а прописью — `два миллиона сорок    │
│  пять тысяч` (разница 405 000 сом). При этом с нас требуют 100% предоплату, а Поставщик оставляет за собой      │
│  право поднять цену уже после нашей оплаты, если курс доллара вырастет всего на 3%.                             │
│  2. **Безнаказанные задержки и отсутствие гарантий (п. 4.3, п. 5.2, п. 6.2, п. 6.4):**                          │
│     Поставщик вправе задержать поставку на 60 дней без штрафов и объяснений. На проверку товара нам дают всего  │
│  1 рабочий день, после чего претензии по скрытому браку не принимаются. Ответственность Поставщика за срыв      │
│  сроков мизерная (0,01%, максимум 2%) и в целом ограничена 5% от суммы, тогда как с нас за день просрочки       │
│  требуют грабительские 0,5% (более 180% годовых).                                                               │
│  3. **Судебные споры в Дубае (п. 10.2):**                                                                       │
│     Все разногласия двух бишкекских компаний переданы в Международный арбитраж Дубая (ОАЭ) на английском        │
│  языке. Расходы на перелеты, пошлины и иностранных юристов в случае спора многократно превысят стоимость        │
│  самого договора.                                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Что обязательно исправить перед подписанием                                                             │
│  - **Сумма договора (п. 3.2):** Устранить расхождение,

In [16]:
risks_table(e2)

,clause,severity,category,explanation
0,2.3,высокий,цена,Позволяет Поставщику требовать доплату даже по...
1,3.1,высокий,оплата,100% предоплата создает высокий финансовый рис...
2,3.1,средний,оформление и подписи,Рукописная правка сокращает срок оплаты до 1 д...
3,3.2,высокий,цена,Критическое расхождение между суммой цифрами (...
4,3.4,средний,оплата,Покупатель несет риск задержек межбанковских п...
5,4.3,высокий,поставка и приёмка,Поставщик может безнаказанно задерживать поста...
6,5.2,высокий,поставка и приёмка,Один день объективно недостаточен для проверки...
7,6.1,высокий,ответственность,Неустойка для Покупателя необоснованно завышен...
8,6.2,высокий,ответственность,Грубая диспропорция ответственности: неустойка...
9,6.4,высокий,ответственность,Ограничение ответственности 5% не позволит Пок...


## Шаг 5. Сравнение с ручным анализом
Заполните таблицу (двойной щелчок по этой ячейке). Сравнивайте с **листом вашей группы** и эталоном (его даст преподаватель после занятия).

| | Группа (один) | E1 (цифровой) | E2 (скан + Vision) |
|---|---|---|---|
| Найдено рисков из эталона |14 | 16| 16|
| Пропущено | 2| 0|0 |
| Лишнее (не риск или выдумано) |0 |0 | 0|
| Время | 46 минут|302.2 секунд | 711.0 секунд|
| Токены | — |69 963 | 137 361|
| Заметили рукописную правку? | да| нет | да|
| Заметили отсутствие подписи поставщика? | да| да| да|



In [19]:
# Скачать результаты для отчёта
!zip -qr results.zip out
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>